# 🧪 Lab 2 : Gouvernance GCP, IAM, Tarification & Garde-fous FinOps

*(Inspiré des labs officiels Google Cloud Skills Boost : "IAM Options & Roles" et "Google Cloud Billing & Cost Management")*

Dans ce lab pratique, vous allez explorer les principes fondamentaux de la gouvernance sur **Google Cloud Platform (GCP)** : la hiérarchie des ressources, la gestion des accès via **IAM (Identity and Access Management)** et les mécanismes de protection financière (**FinOps**, **quotas** et **garde-fous par requête**).

### 🎯 Objectifs :
1. **Inspecter l'organisation GCP** et la configuration du projet actif via la CLI `gcloud`.
2. **Auditer la stratégie IAM** et expérimenter le **Principe du Moindre Privilège** (*Least Privilege*).
3. **Vérifier la Résidence des Données (RGPD)** et comprendre l'impact de la colocalisation entre Cloud Storage et BigQuery.
4. **Tester les garde-fous de facturation BigQuery (`maximum_bytes_billed`)** pour bloquer automatiquement toute requête trop coûteuse avant exécution (coût $0).
5. **Examiner les quotas et budgets de sécurité** appliqués sur l'environnement de cours.

---


> 🧠 **Rappel du Cours - Hiérarchie des Ressources** :
> GCP organise les ressources selon la hiérarchie suivante :
> `Organisation` ➔ `Dossiers (Folders)` ➔ `Projet` ➔ `Ressources (BigQuery, GCS, IAM)`.
> **Toute ressource doit impérativement appartenir à un Projet.** C'est au niveau du Projet que la facturation est rattachée.

---


## 0. Configuration de l'environnement CLI (Cloud Shell / VS Code Terminal)

Exécutez la cellule de code ci-dessous pour identifier votre identité active et vérifier les propriétés du projet GCP du cours :


In [ ]:
# 1. Vérifier votre compte utilisateur Google Cloud authentifié
gcloud auth list

In [ ]:
# 2. Récupérer l'ID du projet GCP actif
export PROJECT_ID=$(gcloud config get-value project)
echo "Projet GCP actif : $PROJECT_ID"

In [ ]:
# 3. Afficher les métadonnées de base du projet
gcloud projects describe $PROJECT_ID --format="table(projectId, name, projectNumber, lifecycleState)"

## 1. Audit IAM & Expérimentation du Moindre Privilège (Least Privilege)

Le composant **IAM** contrôle **"Qui"** (Identité / Compte utilisateur) a le droit d'effectuer **"Quelle action"** (Rôle / Permissions) sur **"Quelle ressource"**.

### Étape 1.1 : Inspecter les rôles attribués à votre compte via la CLI

Exécutez la cellule ci-dessous pour lister la politique IAM attribuée à votre compte sur le projet :

```bash
export PROJECT_ID=$(gcloud config get-value project)

USER_EMAIL=$(gcloud config get-value account)
```

In [ ]:

# Afficher les rôles IAM spécifiquement accordés à votre compte sur ce projet
gcloud projects get-iam-policy $PROJECT_ID \
    --flatten="bindings[].members" \
    --format="table(bindings.role)" \
    --filter="bindings.members:$USER_EMAIL"

### Étape 1.2 : Tester la sécurité dans la Console Graphique GCP (UI)

1. Rendez-vous dans le menu principal (☰) ➔ **IAM & Admin** ➔ **IAM**.
2. Dans la barre de recherche des membres, filtrez avec votre adresse e-mail d'étudiant.
3. Observez vos rôles attribués (ex: `BigQuery User`, `Viewer`, etc.).
4. **Test d'élévation de privilèges** (Sécurité) :
   - Cliquez sur l'icône **Crayon** (Modifier le membre) à côté de votre nom ou sur **Accorder l'accès** (*Grant Access*) en haut de la page.
   - Essayez de vous attribuer le rôle **Propriétaire** (*Owner*) ou **Administrateur IAM** (*IAM Admin*).
   - Cliquez sur **Enregistrer**.

🔍 **Observation attendue** :
Un message d'erreur rouge s'affiche : *« Permission requise resourcemanager.projects.setIamPolicy manquante »*.
👉 **Pourquoi ?** Votre compte est restreint selon le **Principe du Moindre Privilège** (*Least Privilege*) : vous disposez des accès nécessaires pour exécuter des requêtes analytiques et créer des datasets, mais pas pour modifier les règles de sécurité du projet.


**Une fois que vous avez terminé signalez, vous auprès du formateur**

## 2. Conformité & Résidence des Données (Data Residency & RGPD)

Sur Google Cloud, chaque ressource de stockage (Dataset BigQuery ou Bucket Cloud Storage) réside dans une localisation géographique précise :
- **Régionale** (ex: `europe-west9` pour Paris, `europe-west1` pour la Belgique).
- **Multi-régionale** (ex: `EU` pour l'Union Européenne, `US` pour les États-Unis).

> ⚠️ **Règle d'or de colocalisation BigQuery** :
> - **BigQuery ➔ BigQuery (Inter-datasets)** : Les requêtes, jointures ou copies de données entre des tables situées dans des régions différentes (**ex: croiser un dataset BigQuery situé aux US avec un dataset situé en EU**) sont **strictement interdites par le moteur de calcul**.
> - **GCS ➔ BigQuery (Chargement)** : GCP autorise l'import de fichiers bruts depuis un bucket Cloud Storage distant vers BigQuery, ce qui explique pourquoi charger un fichier CSV depuis un bucket GCS situé aux US vers BigQuery EU fonctionne.

---

### 🛠️ Étape 2.1 : Pas à pas — Créer le dataset `bronze_landing` en région `EU` dans la Console BigQuery (UI)

Pour préparer l'exercice d'incompatibilité de région, créez d'abord le dataset `bronze_landing` configuré dans l'Union Européenne (`EU`) via la console Web :

1. **Accéder à la console BigQuery** :
   - Rendez-vous sur votre navigateur à l'adresse : 👉 **[console.cloud.google.com/bigquery](https://console.cloud.google.com/bigquery)**
2. **Ouvrir le formulaire de création de dataset** :
   - Dans le volet de gauche **Explorateur** (*Explorer*), localisez l'ID de votre projet GCP (ex: `lasalle-big-data`).
   - Cliquez sur les **3 petits points verticaux (`⋮`)** situés à droite de l'ID du projet, puis sélectionnez **`Créer un jeu de données`** (*Create dataset*).
3. **Configurer les paramètres du dataset** :
   - **ID du jeu de données** (*Dataset ID*) : Saisissez **`bronze_landing`**
   - **Emplacement des données** (*Data location*) : Sélectionnez **`EU (plusieurs régions dans l'Union européenne)`**
   - *(Configuration par défaut)* : Laissez les autres options inchangées (Expiration des tables : Jamais).
4. **Valider la création** :
   - Cliquez sur le bouton bleu **`CRÉER UN JEU DE DONNÉES`** (*Create dataset*).
5. 📌 **Vérification** : Déroulez l'arborescence sous votre projet dans le volet Explorateur : le dataset `bronze_landing` est désormais disponible.

---

### 🧪 Étape 2.2 : Tester l'incompatibilité de région dans BigQuery (Cross-Region Error)

Maintenant que votre dataset `bronze_landing` est créé en région `EU`, tentez de copier ou de joindre des données provenant directement du jeu de données public `bigquery-public-data.thelook_ecommerce` qui est hébergé aux États-Unis (`US`) :

Copiez le code SQL ci-dessous et exécutez-le dans votre éditeur BigQuery :

In [ ]:
-- ⚠️  REQUÊTE INTER-RÉGIONS (Source US -> Dataset Cible EU)
CREATE OR REPLACE TABLE `bronze_landing.orders_eu_copy` AS
SELECT 
    order_id,
    user_id,
    status,
    created_at
FROM `bigquery-public-data.thelook_ecommerce.orders` -- Table publique située en région US !
LIMIT 100;

🔍 **Observation attendue dans la Console BigQuery** :
BigQuery refuse catégoriquement d'exécuter la requête et affiche immédiatement l'erreur de colocalisation :
```text
Cannot run a query job in location EU that references dataset bigquery-public-data:thelook_ecommerce in location US.
```

👉 **Explication technique** :
- Le dataset cible `bronze_landing` réside en **EU**, l'exécuteur du job BigQuery attribue donc des ressources de calcul (*Slots*) situées en Europe.
- La table source `bigquery-public-data.thelook_ecommerce.orders` réside physiquement dans les datacenters de la région **US**.
- Le moteur BigQuery interdit d'exécuter des calculs en Europe en lisant directement le stockage américain pour des raisons de coût réseau, de latence et de **conformité RGPD** sur la résidence des données.

En revanche, nous pouvons copier les données dans notre Dataset.

__Effectuer une requête deviens donc possible:__

_Il y a eu ici un mouvement de la donnée (COPY). On change donc la résidence de la donnée._
_Nous ne sommes pas sur une simple lecture de donnée située dans une autre région_

In [ ]:
CREATE OR REPLACE EXTERNAL TABLE `bronze_landing.ext_us_sample` -- Schéma EU créé au lab précédent
OPTIONS (
  format = 'CSV',
  uris = ['gs://cloud-samples-data/bigquery/us-states/us-states.csv']       --le .csv est situé sur un bucket aux US
);

In [ ]:

-- 2. Exécutez une requête sur cette table :
SELECT * FROM `bronze_landing.ext_us_sample` LIMIT 10;

## 3. Garde-fous FinOps & Contrôle Strict des Coûts (`maximum_bytes_billed`)

Sur BigQuery, le modèle tarifaire à la requête (*On-Demand*) gère les coûts au volume scanné (~6$ par Téraoctet scanné).
Afin d'empêcher l'exécution accidentelle de requêtes gigantesques (ex: scanner plusieurs Terabytes par erreur avec un `SELECT *`), BigQuery propose une option de sécurité fondamentale : **`maximum_bytes_billed`**.

### Comment ça marche ?
Lorsque vous définissez `maximum_bytes_billed = X` :
1. BigQuery effectue un **Dry Run** automatique et gratuit avant d'exécuter la requête.
2. Si l'estimation scannée dépasse la valeur $X$, BigQuery **annule immédiatement la requête**.
3. **Coût facturé = 0,00 €**.

--- 

### Étape 3.1 : Tester le Garde-fou via la CLI `bq`

Exécutez la cellule ci-dessous pour lancer une requête sur la table volumineuse des dépôts GitHub (`bigquery-public-data.samples.github_nested`, ~2.5 Go par colonne), en fixant un garde-fou strict à **10 Mo** (`10000000` octets) :

In [ ]:
# Exécution avec un garde-fou strict de 10 Mo (doit rejeter l'exécution sans facturation)
bq query \
  --use_legacy_sql=false \
  --maximum_bytes_billed=10000000 \
  "SELECT repository.name, COUNT(1) FROM \`bigquery-public-data.samples.github_nested\` GROUP BY 1"


🔍 **Observation attendue dans la sortie de cellule** :
```text
Error in query string: Error processing job '...': Query exceeded limit for bytes billed: 10000000. Provided 10000000, total scannable bytes were 2542109811.
```
👉 **Explication** : Le serveur BigQuery a stoppé le job **avant même qu'il ne démarre**. Aucune ressource CPU n'a été consommée, aucun centime n'a été dépensé !



## 4. Exercice Pratique : Auditer les Quotas & la Protection FinOps du Cours 🏋️

### Énoncé :

1. En vous appuyant sur la table `bigquery-public-data.wikipedia.pageviews_2025` (table de plusieurs To) :
   - Rédigez une requête calculant le nombre total de vues par titre de page dans le wikipedia du pays de votre choix sur le mois de Janvier 2025.
   - **Avant d'exécuter** : Observez l'estimation du Dry Run (coche verte dans la console UI).
   - Si le volume scanné dépasse votre garde-fou (ex: 1 Go), ajustez votre clause `WHERE` (par exemple en filtrant sur une date) pour faire passer le volume sous le seuil maximal de 1 Go autorisé !

2. **Découverte des Quotas Système GCP** :
   - Allez dans le menu GCP (☰) ➔ **IAM & Admin** ➔ **Quotas et limites système**.
   - Dans le filtre de recherche, tapez `BigQuery API`.
   - Observez le quota **Query usage per day per user**. Quel est-il? 

---


## ✅ Checkpoint de Validation du Lab

- [ ] J'ai identifié l'ID de mon projet GCP et vérifié mes rôles IAM attribués avec `gcloud` et la Console.
- [ ] J'ai vérifié par l'expérience que mon compte ne peut pas élever ses propres privilèges (Principe du Moindre Privilège).
- [ ] J'ai créé le dataset `bronze_landing` en région `EU` via la Console BigQuery.
- [ ] J'ai constaté la règle stricte de colocalisation inter-datasets BigQuery (`EU` vs `US`) en tentant une copie CTAS depuis `thelook_ecommerce`.
- [ ] J'ai testé l'option de sécurité `--maximum_bytes_billed` et vérifié que BigQuery stoppe automatiquement à 0 € les requêtes trop volumineuses.
- [ ] J'ai consulté les quotas d'utilisation BigQuery dans le panneau IAM & Admin.